In [1]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import datetime
import random

df = pd.read_csv(
    'Timeseries_33.967_-6.843_SA3_75kWp_crystSi_14_30deg_8deg_2014_2014.csv', 
    skiprows=10, 
    usecols=[0, 1], 
    skipfooter=11, 
    engine='python'
)
df['time'] = pd.to_datetime(df['time'], format='%Y%m%d:%H%M')

pv_peak_production = 75.900
step_time = 3
timestep = 60 / step_time
days_of_experiment = 365
nb_timesteps = int(round(timestep * 24))
df['daytime'] = df['time'].apply(lambda x: datetime.datetime.strptime(f'{x.day}/{x.month}/{x.year}', '%d/%m/%Y'))
renewable = np.zeros((nb_timesteps + 2))
timestep = int(round(timestep)) if (timestep > 1) else 1
days = list(range(0, days_of_experiment))
random.shuffle(days)
X, Y = [], []

# Define window sizes
input_window = 5  # Number of timesteps for X
output_window = 1  # Number of timesteps for Y

for i, d in enumerate(days):
    df_day_selected = df[df['daytime'] == df['daytime'][d]].copy()
    df_day_selected = df_day_selected.reset_index(drop=True)
    df_day_selected.loc[len(df_day_selected)] = [df_day_selected['time'][len(df_day_selected) - 1] + datetime.timedelta(minutes=60), 0, 0]
    df_resampled = df_day_selected.resample(f'{step_time}T', on='time').mean()
    renewable[0], renewable[-1] = -1, -1
    renewable[1:nb_timesteps+1] = (df_resampled['P'].interpolate().to_numpy() / 1000 / pv_peak_production).tolist()[:-1]
    for j in range(len(renewable) - input_window):
        X.append(renewable[j:j + input_window])  # Input: Sequence of 5 timesteps
        Y.append(renewable[j + input_window:j + input_window + output_window])  # Output: Next 3 timesteps


df = pd.read_csv(
    'Timeseries_48.857_2.352_SA3_75kWp_crystSi_14_38deg_-6deg_2014_2014.csv', 
    skiprows=10, 
    usecols=[0, 1], 
    skipfooter=11, 
    engine='python'
)
df['time'] = pd.to_datetime(df['time'], format='%Y%m%d:%H%M')
df['daytime'] = df['time'].apply(lambda x: datetime.datetime.strptime(f'{x.day}/{x.month}/{x.year}', '%d/%m/%Y'))
days = list(range(0, days_of_experiment))
random.shuffle(days)

# Define window sizes
input_window = 5  # Number of timesteps for X
output_window = 1  # Number of timesteps for Y

for i, d in enumerate(days):
    df_day_selected = df[df['daytime'] == df['daytime'][d]].copy()
    df_day_selected = df_day_selected.reset_index(drop=True)
    df_day_selected.loc[len(df_day_selected)] = [df_day_selected['time'][len(df_day_selected) - 1] + datetime.timedelta(minutes=60), 0, 0]
    df_resampled = df_day_selected.resample(f'{step_time}T', on='time').mean()
    renewable[0], renewable[-1] = -1, -1
    renewable[1:nb_timesteps+1] = (df_resampled['P'].interpolate().to_numpy() / 1000 / pv_peak_production).tolist()[:-1]
    for j in range(len(renewable) - input_window):
        X.append(renewable[j:j + input_window])  # Input: Sequence of 5 timesteps
        Y.append(renewable[j + input_window:j + input_window + output_window])  # Output: Next 3 timesteps

X_train, X_valid, Y_train, Y_valid = train_test_split(X, Y, test_size=0.2, random_state=0)

print(len(X), len(Y))
print(len(X_train), len(Y_train))
print(len(X_valid), len(Y_valid))

348210 348210
278568 278568
69642 69642


In [5]:
from xgboost import XGBRegressor
import xgboost as xgb

# Create a XGBoost model
modelXG = XGBRegressor(objective= 'reg:squarederror',  # objective function for regression problems
    learning_rate= 0.1,  # learning rate
    max_depth= 3,  # maximum depth of tree
    n_estimators= 100  # number of trees in the model
)

# Fit the model to the training data
resultsXG = modelXG.fit(X_train, Y_train)

# Perform forecast for validation data
predictionsXG = resultsXG.predict(X_valid)
print(predictionsXG.shape)

# Calculate errors for current window
mse = mean_squared_error(Y_valid, predictionsXG)
mae = mean_absolute_error(Y_valid, predictionsXG)
p_mae = (mae / 250) * 100
rmse = np.sqrt(mse)
p_rmse = (rmse / 250) * 100

print("Root Mean Square Error:", rmse)
print('P RMSE: %f' % p_rmse)
print('------')
print('MSE: %f' % mse)
print('------')
print('MAE: %f' % mae)
print('P MAE: %f' % p_mae)

modelXG.save_model("xgboost_pv_model.json")

(69642,)
Root Mean Square Error: 0.045865407148229495
P RMSE: 0.018346
------
MSE: 0.002104
------
MAE: 0.004321
P MAE: 0.001728


In [2]:
import torch
import torch.nn as nn

class CNN_LSTM_Model(nn.Module):
    def __init__(self):
        super(CNN_LSTM_Model, self).__init__()
        
        # 1D Convolution Layer
        self.conv1d = nn.Conv1d(
            in_channels=1,       # Input channels (for single feature, it's 1)
            out_channels=32,     # Number of filters
            kernel_size=9,      # Kernel size (10 initially)
            stride=1,            # Stride
            padding='same'       # 'causal' padding in TensorFlow is equivalent to 'same' padding here
        )
        
        # Activation (ReLU)
        self.relu = nn.ReLU()
        
        # Max Pooling Layer
        self.maxpool = nn.MaxPool1d(kernel_size=2)
        
        # LSTM Layers
        self.lstm1 = nn.LSTM(
            input_size=32,       # Input size is the number of filters from Conv1D
            hidden_size=32,      # Number of LSTM units
            batch_first=True,    # Batch dimension comes first
            bidirectional=False, # Unidirectional LSTM
            num_layers=1         # Single layer
        )
        self.lstm2 = nn.LSTM(
            input_size=32,
            hidden_size=32,
            batch_first=True,
            bidirectional=False,
            num_layers=1
        )
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(32, 16)  # Dense layer with 16 units
        self.fc2 = nn.Linear(16, 1)  # Output layer with 1 unit

    def forward(self, x):
        # x shape: (batch_size, window_size, 1)
        
        # Conv1D expects input in the shape (batch_size, in_channels, seq_length)
        x = x.permute(0, 2, 1)  # Change shape to (batch_size, 1, window_size)
        x = self.conv1d(x)      # Apply Conv1D
        x = self.relu(x)        # Apply ReLU activation
        x = self.maxpool(x)     # Apply MaxPooling
        
        # LSTM expects input in the shape (batch_size, seq_length, input_size)
        x = x.permute(0, 2, 1)  # Change shape to (batch_size, seq_length, input_size)
        
        x, _ = self.lstm1(x)    # First LSTM layer
        x, _ = self.lstm2(x)    # Second LSTM layer
        
        # Take only the last output of the sequence for further layers
        x = x[:, -1, :]         # Shape: (batch_size, hidden_size)
        
        x = self.fc1(x)         # Fully connected layer 1
        x = self.relu(x)        # Apply ReLU activation
        x = self.fc2(x)         # Fully connected layer 2
        
        return x

# Example usage:
model = CNN_LSTM_Model()

# Define loss function and optimizer
criterion = nn.MSELoss()  # Mean Squared Error Loss
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 400
batch_size = 32

# Convert X_train and Y_train to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)
Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

# Training
model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    # Forward pass
    predictions = model(X_train_tensor)  # Model outputs (batch_size, 1)
    
    # Compute loss
    loss = criterion(predictions, Y_train_tensor)
    
    # Backward pass and optimization
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}")

# Evaluation
X_valid_tensor = torch.tensor(X_valid, dtype=torch.float32).unsqueeze(-1)
model.eval()
with torch.no_grad():
    # Perform predictions
    predictions = model(X_valid_tensor).squeeze().numpy()  # Convert to numpy array

# Calculate errors
mse = mean_squared_error(Y_valid, predictions)
mae = mean_absolute_error(Y_valid, predictions)
p_mae = (mae / 250) * 100
rmse = np.sqrt(mse)
p_rmse = (rmse / 250) * 100

# Print metrics
print("Root Mean Square Error:", rmse)
print('P RMSE: %f' % p_rmse)
print('------')
print('MSE: %f' % mse)
print('------')
print('MAE: %f' % mae)
print('P MAE: %f' % p_mae)

/tmp/ipykernel_253921/2294874994.py:79: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:230.)
  X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1)


Epoch 1/400, Loss: 0.008088571950793266
Epoch 2/400, Loss: 0.007450432516634464
Epoch 3/400, Loss: 0.006865199655294418
Epoch 4/400, Loss: 0.006333973724395037
Epoch 5/400, Loss: 0.005976437591016293
Epoch 6/400, Loss: 0.005644944496452808
Epoch 7/400, Loss: 0.005340228788554668
Epoch 8/400, Loss: 0.00506217684596777
Epoch 9/400, Loss: 0.004810328595340252
Epoch 10/400, Loss: 0.004590739496052265
Epoch 11/400, Loss: 0.004396301228553057
Epoch 12/400, Loss: 0.004223683848977089
Epoch 13/400, Loss: 0.00409142067655921
Epoch 14/400, Loss: 0.003998593892902136
Epoch 15/400, Loss: 0.003913384396582842
Epoch 16/400, Loss: 0.0038367395754903555
Epoch 17/400, Loss: 0.003769091796129942
Epoch 18/400, Loss: 0.003712310688570142
Epoch 19/400, Loss: 0.0036823120899498463
Epoch 20/400, Loss: 0.0036557603161782026
Epoch 21/400, Loss: 0.003632558509707451
Epoch 22/400, Loss: 0.003612467786297202
Epoch 23/400, Loss: 0.0035952506586909294
Epoch 24/400, Loss: 0.0035806780215352774
Epoch 25/400, Loss: 0.

Epoch 197/400, Loss: 0.0021881142165511847
Epoch 198/400, Loss: 0.002181882970035076
Epoch 199/400, Loss: 0.0021712733432650566
Epoch 200/400, Loss: 0.0021600087638944387
Epoch 201/400, Loss: 0.0021492280066013336
Epoch 202/400, Loss: 0.002138417446985841
Epoch 203/400, Loss: 0.0021280357614159584
Epoch 204/400, Loss: 0.002119639189913869
Epoch 205/400, Loss: 0.0021143120247870684
Epoch 206/400, Loss: 0.0021117315627634525
Epoch 207/400, Loss: 0.0021108000073581934
Epoch 208/400, Loss: 0.002110759261995554
Epoch 209/400, Loss: 0.0021114489063620567
Epoch 210/400, Loss: 0.0021128826774656773
Epoch 211/400, Loss: 0.002114789793267846
Epoch 212/400, Loss: 0.002116561634466052
Epoch 213/400, Loss: 0.002117676893249154
Epoch 214/400, Loss: 0.00211799843236804
Epoch 215/400, Loss: 0.002117711817845702
Epoch 216/400, Loss: 0.0021170377731323242
Epoch 217/400, Loss: 0.0021160405594855547
Epoch 218/400, Loss: 0.002114672213792801
Epoch 219/400, Loss: 0.0021129436790943146
Epoch 220/400, Loss: 0

Epoch 390/400, Loss: 0.0020966408774256706
Epoch 391/400, Loss: 0.002096627140417695
Epoch 392/400, Loss: 0.0020966138690710068
Epoch 393/400, Loss: 0.0020966005977243185
Epoch 394/400, Loss: 0.002096587559208274
Epoch 395/400, Loss: 0.0020965752191841602
Epoch 396/400, Loss: 0.0020965617150068283
Epoch 397/400, Loss: 0.002096549142152071
Epoch 398/400, Loss: 0.0020965440198779106
Epoch 399/400, Loss: 0.002096537733450532
Epoch 400/400, Loss: 0.0020965361036360264
Root Mean Square Error: 0.045981227610816196
P RMSE: 0.018392
------
MSE: 0.002114
------
MAE: 0.005286
P MAE: 0.002115


In [3]:
from sklearn.svm import SVR
import numpy as np

# Create an SVR model with default parameters
modelSVR = SVR(kernel='rbf', C=1.0, epsilon=0.01, gamma='auto')  # RBF kernel is commonly used

# Fit the model to the training data
resultsSVR = modelSVR.fit(X_train, np.array(Y_train).reshape((len(Y_train),)))

# Perform forecast for validation data
predictionsSVR = resultsSVR.predict(X_valid)
print(predictionsSVR.shape)

# Calculate errors for the current window
mse = mean_squared_error(Y_valid, predictionsSVR)
mae = mean_absolute_error(Y_valid, predictionsSVR)
p_mae = (mae / 250) * 100
rmse = np.sqrt(mse)
p_rmse = (rmse / 250) * 100

print("Root Mean Square Error:", rmse)
print('P RMSE: %f' % p_rmse)
print('------')
print('MSE: %f' % mse)
print('------')
print('MAE: %f' % mae)
print('P MAE: %f' % p_mae)

(69642,)
Root Mean Square Error: 0.0463150940624111
P RMSE: 0.018526
------
MSE: 0.002145
------
MAE: 0.010416
P MAE: 0.004166
